# 25. 93쪽 항목 2 확장 적재 (Chunk b05·b06, 행동 a05·a06, 관계 10개)

- 승인: 사용자 "a로 진행" (개념 EXCLUDE_FROM_SCHOOL, END_EXCLUSION 그대로).
- 검사 사전 확인: 충돌 0건(같은 ID 없음), 끝점 각 1개, Codex 검토 CHANGE 반영(관계 10개).
- 안전: 한 트랜잭션, CREATE 전용, 실패 시 전부 되돌림. 임베딩은 트랜잭션 밖.

In [1]:
import sys
sys.path.insert(0, "../../src")
from rag.prevention_legal.load_graph import add_extension
result = add_extension()
print("적재 결과 (검사 통과):", result)

적재 결과 (검사 통과): {'chunks_added': 2, 'actions_added': 2, 'measles_93_actions': 6, 'relations_added': 10}


In [2]:
from rag.prevention_legal.neo4j_conn import connect
d = connect()
with d.session() as s:
    print("새 Chunk:", s.run("MATCH (c:Chunk) WHERE c.chunk_id IN ['kdca_measles_2026:p0093:b05','kdca_measles_2026:p0093:b06'] RETURN count(c) AS n").single()["n"])
    print("93쪽 행동:", s.run("MATCH (a:SchoolAction) WHERE a.action_id STARTS WITH 'kdca_measles_2026:p0093' RETURN count(a) AS n").single()["n"])
    print("a05·a06 관계:", s.run("MATCH (a:SchoolAction)-[r]->(x) WHERE a.action_id IN ['kdca_measles_2026:p0093:a05','kdca_measles_2026:p0093:a06'] RETURN a.action_id AS a, type(r) AS t, coalesce(x.chunk_id, x.disease_id, x.role_id, x.concept_id) AS to ORDER BY a, t").data())
    print("전체 Chunk:", s.run("MATCH (c:Chunk) RETURN count(c) AS n").single()["n"])
d.close()

새 Chunk: 2
93쪽 행동: 6


a05·a06 관계: [{'a': 'kdca_measles_2026:p0093:a05', 't': 'DONE_BY', 'to': 'SCHOOL_TEACHER_IN_CHARGE'}, {'a': 'kdca_measles_2026:p0093:a05', 't': 'FOR_DISEASE', 'to': 'MEASLES'}, {'a': 'kdca_measles_2026:p0093:a05', 't': 'INSTANCE_OF', 'to': 'EXCLUDE_FROM_SCHOOL'}, {'a': 'kdca_measles_2026:p0093:a05', 't': 'SUPPORTED_BY', 'to': 'kdca_measles_2026:p0093:b05'}, {'a': 'kdca_measles_2026:p0093:a06', 't': 'DONE_BY', 'to': 'SCHOOL_TEACHER_IN_CHARGE'}, {'a': 'kdca_measles_2026:p0093:a06', 't': 'FOR_DISEASE', 'to': 'MEASLES'}, {'a': 'kdca_measles_2026:p0093:a06', 't': 'INSTANCE_OF', 'to': 'END_EXCLUSION'}, {'a': 'kdca_measles_2026:p0093:a06', 't': 'SUPPORTED_BY', 'to': 'kdca_measles_2026:p0093:b06'}]


전체 Chunk: 1569


In [3]:
from neo4j.exceptions import ConstraintError
try:
    add_extension()
    print("재실행이 성공함 (문제)")
except ConstraintError as e:
    print("재실행 실패 (기대):", type(e).__name__)
d = connect()
with d.session() as s:
    print("전체 Chunk (재실행 후):", s.run("MATCH (c:Chunk) RETURN count(c) AS n").single()["n"])
    print("93쪽 행동 (재실행 후):", s.run("MATCH (a:SchoolAction) WHERE a.action_id STARTS WITH 'kdca_measles_2026:p0093' RETURN count(a) AS n").single()["n"])
d.close()

재실행 실패 (기대): ConstraintError


전체 Chunk (재실행 후): 1569
93쪽 행동 (재실행 후): 6
